# Modèle Autorégressif GPT — Génération de critiques de vins

Ce notebook implémente un **modèle de langage autorégressif** de type GPT (Generative Pre-trained Transformer) entraîné sur des critiques de vins. Le modèle apprend à prédire le mot suivant étant donné le contexte précédent, ce qui lui permet ensuite de **générer du texte** de manière cohérente à partir d'une amorce.

L'architecture repose sur un **bloc Transformer** avec attention causale (masquée), des embeddings de tokens et de positions, et une tête de classification sur le vocabulaire.

## 1. Initialisation, Chargement des Données et Prétraitement

Cette étape prépare le terrain pour l'entraînement de notre modèle de traitement du langage naturel (NLP). Elle se charge d'importer les outils nécessaires, de télécharger notre jeu de données (des critiques de vins), de définir les réglages globaux du modèle (hyperparamètres), et de créer un outil capable de transformer notre texte brut en nombres que le modèle pourra comprendre.

* **Imports et Hyperparamètres :** Nous importons `torch` (PyTorch) pour le Deep Learning, ainsi que divers utilitaires. Nous définissons ensuite les constantes de notre futur modèle (comme la taille du vocabulaire `VOCAB_SIZE`, la dimension des embeddings `EMBEDDING_DIM`, et la configuration de l'attention `N_HEADS`). Le code détecte aussi automatiquement si une carte graphique (`cuda`) est disponible pour accélérer les calculs.
* **Acquisition et Formatage :** Nous téléchargeons le dataset "wine-reviews" via l'API Kaggle (`kagglehub`). Nous lisons le fichier JSON et filtrons les données pour retirer les entrées incomplètes. Ensuite, nous concaténons plusieurs informations (pays, province, variété et description) dans une chaîne de caractères unique et formatée pour chaque vin.
* **Le Tokeniseur (`SimpleTokenizer`) :** Les réseaux de neurones ne lisent pas de texte, ils lisent des nombres. Cette classe personnalisée parcourt tout notre texte, compte la fréquence des mots et crée un dictionnaire des 14 998 mots les plus fréquents.
  * L'index `0` est réservé au padding (`<pad>`, pour uniformiser la taille des phrases).
  * L'index `1` est réservé aux mots inconnus (`<unk>`).
  * Sa méthode `.encode()` permet de convertir n'importe quelle phrase en une liste d'index numériques.
* **Nettoyage (`pad_punctuation`) :** Cette fonction ajoute des espaces autour des signes de ponctuation pour s'assurer qu'ils soient traités comme des mots indépendants par le tokeniseur, et passe tout le texte en minuscules pour réduire la complexité du vocabulaire.

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn.functional import cross_entropy
import numpy as np
import re, string
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter
import json
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zynicide/wine-reviews" )
# Parameters
VOCAB_SIZE = 15000
MAX_LEN = 80
EMBEDDING_DIM = 256
N_HEADS = 2
FF_DIM = 256
BATCH_SIZE = 32
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the data
with open(path+"/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]
# Tokenizer
class SimpleTokenizer:
    def __init__(self, texts, vocab_size):
        self.vocab_size = vocab_size
        self.counter = Counter()
        for text in texts:
            self.counter.update(text.split())
        self.vocab = [word for word, _ in self.counter.most_common(vocab_size - 2)]
        self.word2idx = {word: idx + 2 for idx, word in enumerate(self.vocab)}
        self.word2idx["<pad>"] = 0
        self.word2idx["<unk>"] = 1

    def encode(self, text):
        return [self.word2idx.get(word, 1) for word in text.split()]

# Data preparation
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s.lower()


# Create tokenizer
tokenizer = SimpleTokenizer(filtered_data, VOCAB_SIZE)

## 2. Préparation des lots de données (Dataset & DataLoader)

Transformation de notre grande liste de textes en une "chaîne d'assemblage" optimisée pour l'intelligence artificielle. Au lieu de donner tout le texte d'un coup au modèle (ce qui ferait planter l'ordinateur), cette étape prépare un système capable de distribuer les données par petits paquets homogènes, tout en créant la logique d'apprentissage du modèle : "étant donné ce début de phrase, devine le mot suivant".

* **La classe `WineDataset` :** C'est un objet hérité de PyTorch (`Dataset`) qui explique comment lire nos données une par une. Elle contient trois méthodes obligatoires :
  * `__init__` : Stocke nos textes, notre tokeniseur et la longueur maximale autorisée (`max_len`).
  * `__len__` : Renvoie simplement le nombre total de critiques de vins dont nous disposons.
  * `__getitem__` : C'est le cœur de la préparation. Pour un index donné, cette fonction convertit la phrase en nombres. Si la phrase est trop courte, elle ajoute des zéros (le jeton `<pad>`) jusqu'à atteindre la limite fixée. Si elle est trop longue, elle la coupe.
* **La logique d'apprentissage (Input vs Target) :** À la fin de `__getitem__`, la fonction ne renvoie pas une, mais **deux** séquences (sous forme de tenseurs PyTorch).
  * La première (`tokens[:-1]`) est la donnée d'entrée : la phrase à laquelle on a retiré le dernier mot.
  * La seconde (`tokens[1:]`) est la cible à deviner : c'est la même phrase, mais décalée d'un mot vers la droite. C'est ce qui permettra au modèle d'apprendre à prédire le mot suivant.
* **Création des objets :** * `train_dataset` crée concrètement l'objet avec nos données (en appliquant au passage la fonction de nettoyage `pad_punctuation`).
  * `train_loader` (le `DataLoader`) encapsule notre dataset pour automatiser le travail pénible : il va mélanger les données à chaque passage (`shuffle=True`) et les distribuer par lots de 32 (`BATCH_SIZE`) pour accélérer l'entraînement sur la carte graphique.

In [12]:
# Dataset class
class WineDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.texts[idx])[:self.max_len+1]
        padding = [self.tokenizer.word2idx["<pad>"]] * (self.max_len + 1 - len(tokens))
        tokens += padding
        return torch.tensor(tokens[:-1]), torch.tensor(tokens[1:])

train_dataset = WineDataset([pad_punctuation(t) for t in filtered_data], tokenizer=tokenizer, max_len=MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

## 3. Bloc Transformer (`GPTBlock`)

Brique fondamentale du modèle GPT. Ce bloc implémente un **Transformer decoder** simplifié avec attention causale :

- **`MultiheadAttention`** : mécanisme d'attention multi-têtes en mode *self-attention*. Un **masque causal triangulaire inférieur** (`torch.tril`) est généré dynamiquement pour que chaque token ne puisse voir que les tokens précédents — garantissant la propriété autoregressive.
- **Feed-Forward Network (FFN)** : deux couches linéaires avec activation ReLU (Linear → ReLU → Linear), permettant au modèle d'apprendre des transformations non-linéaires complexes.
- **Layer Normalization** (`ln1`, `ln2`) : normalisation appliquée après chaque sous-couche pour stabiliser l'entraînement.
- **Connexions résiduelles** (`x + attn_output`, `x + ffn_output`) : ajout de l'entrée à la sortie de chaque sous-couche (skip connections), facilitant la propagation du gradient.

In [13]:
class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        mask = mask.masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)

        attn_output, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.ln2(x + ffn_output)


## 4. Modèle GPT complet (`GPTModel`)

Assemblage du modèle complet à partir des briques précédentes :

1. **`nn.Embedding(vocab_size, embed_dim)`** : transforme chaque index de token en un vecteur dense de dimension 256.
2. **`nn.Embedding(max_len, embed_dim)`** : **embedding de position** — un vecteur appris est ajouté à chaque token selon sa position dans la séquence, permettant au modèle de tenir compte de l'ordre des mots.
3. **`GPTBlock`** : le bloc Transformer avec attention masquée (défini ci-dessus).
4. **`nn.Linear(embed_dim, vocab_size)`** : couche de sortie qui projette chaque représentation de token vers un vecteur de logits de taille `VOCAB_SIZE` (15 000).

Le `forward` génère les positions `[0, 1, ..., seq_len-1]`, additionne les embeddings de tokens et de positions, passe dans le bloc Transformer, puis renvoie les logits pour chaque position.

In [14]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, num_heads, ff_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.transformer = GPTBlock(embed_dim, num_heads, ff_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)
        return self.fc(x)

## 5. Boucle d'entraînement

Entraînement du modèle sur 100 epochs avec :
- **Optimiseur** : Adam (lr = 0.0001)
- **Fonction de perte** : `CrossEntropyLoss` sur la prédiction du token suivant, avec `ignore_index` sur le token `<pad>` pour ne pas pénaliser les positions de padding

À chaque batch :
1. `x_batch` (séquence d'entrée, tokens 0 à N-1) et `y_batch` (cible, tokens 1 à N) sont envoyés sur le GPU
2. Le modèle produit des logits de forme `[B, seq_len, vocab_size]`
3. Les logits et cibles sont aplatis (`view(-1, ...)`) pour le calcul de la CrossEntropy
4. Rétropropagation et mise à jour des poids

La perte moyenne par epoch est affichée pour suivre la convergence.

In [15]:
# Training loop
# Training
model = GPTModel(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, N_HEADS, FF_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=0.0001)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = cross_entropy(logits.view(-1, VOCAB_SIZE), y_batch.view(-1), ignore_index=tokenizer.word2idx['<pad>'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}")

Epoch 1/100: 100%|██████████| 4060/4060 [01:32<00:00, 43.86it/s]


Epoch 1, Training Loss: 4.0773


Epoch 2/100: 100%|██████████| 4060/4060 [01:47<00:00, 37.70it/s]


Epoch 2, Training Loss: 3.4254


Epoch 3/100: 100%|██████████| 4060/4060 [01:53<00:00, 35.65it/s]


...

Epoch 98, Training Loss: 2.3110


Epoch 99/100: 100%|██████████| 4060/4060 [01:40<00:00, 40.38it/s]


Epoch 99, Training Loss: 2.3097


Epoch 100/100: 100%|██████████| 4060/4060 [01:39<00:00, 40.61it/s]

Epoch 100, Training Loss: 2.3083


## 6. Fonction de génération de texte

Fonction `generate_text` : génère du texte de manière **autoregressive** à partir d'une amorce.

**Processus** :
1. L'amorce est encodée en tokens et tronquée à `MAX_LEN`
2. À chaque itération :
   - Les derniers `MAX_LEN` tokens générés sont passés dans le modèle
   - On extrait les **logits du dernier token** uniquement (`logits[:, -1, :]`)
   - On divise par la **température** (< 1 = plus déterministe, > 1 = plus créatif/aléatoire)
   - `softmax` donne une distribution de probabilité sur le vocabulaire
   - **Échantillonnage** du token suivant via `torch.multinomial`
3. Si le token `<pad>` est généré, la génération s'arrête

Le résultat est converti de tokens vers mots via un dictionnaire inverse `idx2word`.

In [16]:
def generate_text(model, tokenizer, start_prompt, max_tokens=80, temperature=1.0):
    model.eval()
    tokens = tokenizer.encode(start_prompt)
    tokens = tokens[:MAX_LEN]
    generated = tokens.copy()

    with torch.no_grad():
        for _ in range(max_tokens):
            input_tensor = torch.tensor([generated[-MAX_LEN:]], device=DEVICE)
            logits = model(input_tensor)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            if next_token == tokenizer.word2idx['<pad>']:
                break

            generated.append(next_token)

    idx2word = {idx: word for word, idx in tokenizer.word2idx.items()}
    generated_text = ' '.join(idx2word.get(idx, '<unk>') for idx in generated)
    return generated_text


## 7. Génération d'exemples

Utilisation du modèle entraîné pour générer une critique de vin à partir d'une amorce structurée.

L'amorce `"wine review : US"` suit le format des données d'entraînement (`pays : province : variété : description`). Avec une **température de 0.8** (légèrement en dessous de 1), le modèle génère un texte plus cohérent et moins aléatoire qu'avec une température élevée. Jusqu'à 500 tokens sont générés.

In [18]:
# Example usage:
prompt = "wine review : US"
generated_text = generate_text(model, tokenizer, prompt, max_tokens=500, temperature=0.8).strip()
print("Generated text:\n", generated_text)

Generated text:
 wine review : US : <unk> : <unk> : this is a super fresh wine with notes of ripe , peachy fruit and banana . the balance <unk> in the mouth but <unk> <unk> <unk> it <unk> <unk> more new . it <unk> <unk> <unk> balanced and with a powdery , textural feel . <unk> . cucumber and white grapefruit flavors linger on the finish , which <unk> <unk> <unk> <unk> stop anyone . try as a wet stone . drink <unk> . <unk> . now . imported by <unk> . drink now . will <unk> , pair this with a <unk> , <unk> <unk> <unk> . <unk> . <unk> . <unk> <unk> <unk> <unk> <unk> <unk> best 2012 and 2015 ready <unk> . drink now through 2015 . <unk> <unk> drink after a long solo sipping at least a decade of cellaring . <unk> . <unk> <unk> <unk> <unk> any <unk> of what <unk> <unk> <unk> most notable <unk> <unk> spectacular <unk> : <unk> <unk> early 2017 while . <unk> <unk> the two : <unk> . <unk> <unk> , <unk> by the <unk> 2011 . the mixed <unk> <unk> <unk> <unk> always a stylish wine , offering long - a